# Stage 4 Classification and Judging

## Rung 0 - Setup and load the stage-3 handoff

Code block #1: importing stage 3 data

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import config

#Imports of machine leraning tools used right away
import numpy as np 
import matplotlib.pyplot as plt 
from astropy.table import Table 

#loading in catalog
cat = Table.read("../ztfdata/catalog/ztf_20180322273264_000468_zr_c03_o_q2_stage3_catalog.ecsv", format="ascii.ecsv")

print(len(cat))
print(cat.colnames)

Code Block 2: looking at data

In [ ]:
for col in ["snr", "elongation", "area"]:
    c = cat[col]
    print(f"{col:12s} min {c.min():7.2f}   med {np.median(c):7.2f}   max {c.max():7.2f}")

n_pos = (cat["sign"] == 1).sum()
n_neg = (cat["sign"] == -1).sum()
n_edge = cat["on_edge"].sum()

print()
print(f"sign:     {n_pos} positive (+1, appeared/brightened) | {n_neg} negative (-1, faded)")
print(f"on_edge:  {n_edge} of {len(cat)} candidates touch a stamp edge")


## Rung 1: Installing missing applications
includes (done in terminal btw. no need for a code block, but good to explain it here just for future reference):
- scikit-learn
- astroquery
- imbalanced-learn

## Rung 2: Configure ML for classification
SkyBoT cross match --> free astroid labels

Code Block 3: get observation time from header

In [ ]:
import glob
from astropy.io import fits

# the raw sciimg is under ztfdata/sci/<year>/<mmdd>/<fracday>/...
matches = glob.glob("../ztfdata/sci/**/*20180322273264*sciimg.fits", recursive=True)
print("found:", matches)

hdr = fits.open(matches[0])[0].header
for key in ["OBSJD", "OBSMJD", "MJD-OBS", "JD", "DATE-OBS", "FILTER", "FIELDID"]:
    if key in hdr:
        print(f"{key:10s} = {hdr[key]}")



Code Block 4: Actual SkyBoT query

In [ ]:
from astroquery.imcce import Skybot
from astropy.coordinates import SkyCoord
from astropy.time import Time
import astropy.units as u

field_center = SkyCoord(ra=149.8*u.deg, dec=2.2*u.deg)
epoch = Time(hdr["OBSJD"], format="jd")

try:
    sbt = Skybot.cone_search(field_center, rad=0.15*u.deg, epoch=epoch)
    print(f"SkyBoT found {len(sbt)} known solar-system object(s) in frame:")
    print(sbt["Name", "RA", "DEC", "V"])
except RuntimeError as e:
    sbt = None
    print("SkyBoT found 0 known objects in this field at this time.")
    print("   (Expected on a random equatorial field — this is a valid result, not an error.)")
    print("   raw message:", e)


Code Block 5: Cross match SkyBoT with our detections

In [ ]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u

#detections as a SkyCoord array
det_coords = SkyCoord(ra=cat["ra"]*u.deg, dec=cat["dec"]*u.deg)

# SkyBoT asteroids coordinates
ast_coords = SkyCoord(ra=sbt["RA"], dec=sbt["DEC"])

# for each asteroid, find nearest of our detections and how far
idx, sep2d, _ = ast_coords.match_to_catalog_sky(det_coords)

TOL = 5 * u.arcsec
xmatch = np.zeros(len(cat), dtype=bool)
for name, i, sep in zip(sbt["Name"], idx, sep2d):
    hit = sep < TOL
    if hit:
        xmatch[int(i)] = True
    flag = "MATCH" if hit else "no match" 
    print(f"{name:12s} -> nearest detection #{int(i):2d}, {sep.arcsec:7.1f} arcsec  [{flag}]")

print(f"\n{xmatch.sum()} of {len(cat)} detections cross-matched to a known asteroid.")
